In [53]:
!pip install -q transformers datasets evaluate scikit-learn accelerate peft sentencepiece protobuf
!pip install -U "torchao>=0.16.0" peft transformers datasets evaluate

In [54]:
import os
import random
import numpy as np
import pandas as pd

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_PROJECT"] = "Fact_Checker_Travel"

import torch
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, 
                          TrainingArguments, Trainer, DataCollatorWithPadding, 
                          pipeline, EarlyStoppingCallback)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
from sklearn.metrics import confusion_matrix, classification_report


SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    
    REPORT_TO = "wandb"
    print("W&B attivato.")
except Exception as e:
    print(f"W&B disabilitato. ({e})")
    os.environ["WANDB_DISABLED"] = "true"
    REPORT_TO = "none"

Device: cuda


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


W&B attivato.


In [55]:
from datasets import Dataset, DatasetDict, ClassLabel

df = pd.read_csv('/kaggle/input/datasets/hentoo99/fatcheck/factcheck_dataset_japan.csv')
df = df.rename(columns={"context": "premise", "claim": "hypothesis"})

ID2LABEL = {0: "contradiction", 1: "entailment", 2: "neutral_hard"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

dataset = Dataset.from_pandas(df)
train_test = dataset.train_test_split(test_size=0.3, seed=SEED)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=SEED)
ds = DatasetDict({'train': train_test['train'], 'validation': val_test['train'], 'test': val_test['test']})

rows = []
for split in ["train", "validation", "test"]:
    counts = pd.Series(ds[split]["label"]).value_counts().sort_index()
    row = {ID2LABEL[i]: int(counts.get(i, 0)) for i in ID2LABEL}   
    row["Totale"] = len(ds[split])
    rows.append(row)

dist = pd.DataFrame(rows, index=["train", "validation", "test"])
dist.loc["TOTALE"] = dist.sum()

print("\nConteggio esempi per split e per label:")
print(dist)

train_labels = ds['train']['label']
counts = pd.Series(train_labels).value_counts().sort_index().values.astype(float)
class_weights = torch.tensor(counts.sum() / (NUM_LABELS * counts), dtype=torch.float).to(DEVICE)
print("\n\n⚖️ Pesi calcolati per le classi (Contradiction, Entailment, Neutral):", class_weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels, weight=class_weights.float())
        return (loss, outputs) if return_outputs else loss


Conteggio esempi per split e per label:
            contradiction  entailment  neutral_hard  Totale
train                 211         216           258     685
validation             44          46            57     147
test                   45          40            62     147
TOTALE                300         302           377     979


⚖️ Pesi calcolati per le classi (Contradiction, Entailment, Neutral): [1.082148551940918, 1.0570987462997437, 0.8850129246711731]


In [56]:
model_checkpoint = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples["premise"], 
        examples["hypothesis"], 
        truncation=True, 
        max_length=256
    )

tokenized_ds = ds.map(tokenize_function, batched=True)
colonne_da_rimuovere = [c for c in ["fact_id", "premise", "hypothesis", "claim_type", "__index_level_0__"] if c in tokenized_ds["train"].column_names]
tokenized_ds = tokenized_ds.remove_columns(colonne_da_rimuovere)
tokenized_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

import evaluate
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.logits if hasattr(eval_pred, 'logits') else eval_pred.predictions, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=eval_pred.label_ids)["accuracy"]
    f1 = metric_f1.compute(predictions=preds, references=eval_pred.label_ids, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": f1}

Map:   0%|          | 0/685 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

In [57]:
print("--- CALCOLO BASELINE SUL TEST SET ---")

nli_pipe = pipeline("text-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=0 if DEVICE=="cuda" else -1)

def map_nli_label(label_str):
    if "entailment" in label_str.lower(): return 1
    if "contradiction" in label_str.lower(): return 0
    return 2

zero_shot_preds, true_labels = [], []
for item in ds["test"]:
    text = f"{item['premise']} {nli_pipe.tokenizer.sep_token} {item['hypothesis']}"
    res = nli_pipe(text, truncation=True, max_length=256)[0]
    zero_shot_preds.append(map_nli_label(res['label']))
    true_labels.append(item['label'])

zs_acc = metric_acc.compute(predictions=zero_shot_preds, references=true_labels)["accuracy"]
zs_f1 = metric_f1.compute(predictions=zero_shot_preds, references=true_labels, average="macro")["f1"]
print(f"📊 Baseline (Zero-Shot mDeBERTa) - Accuracy: {zs_acc:.4f}, Macro-F1: {zs_f1:.4f}")

del nli_pipe
torch.cuda.empty_cache()

--- CALCOLO BASELINE SUL TEST SET ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

📊 Baseline (Zero-Shot mDeBERTa) - Accuracy: 0.8980, Macro-F1: 0.8945


In [58]:
print("\n Avvio Ricerca Iperparametri LoRA sul Validation Set (Con Early Stopping)...")

results_log = []
lora_ranks = [4, 16]
learning_rates = [1e-4, 3e-4]

best_f1 = 0
best_config = {}

for r in lora_ranks:
    for lr in learning_rates:
        print(f"\n⚙️ Test -> Rank: {r}, LR: {lr}")
        reset_seeds(SEED)
        lora_config = LoraConfig(
            r=r, 
            lora_alpha=16, 
            lora_dropout=0.1, 
            bias="none", 
            task_type=TaskType.SEQ_CLS,
            target_modules=["query_proj", "key_proj", "value_proj", "output_proj","dense_h_to_4h", "dense_4h_to_h"]
        )
        
        temp_model = AutoModelForSequenceClassification.from_pretrained( model_checkpoint, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
        temp_model = get_peft_model(temp_model, lora_config)
        
        training_args = TrainingArguments(
            output_dir=f"./lora_search_r{r}_lr{lr}", 
            learning_rate=lr,
            per_device_train_batch_size=8, 
            per_device_eval_batch_size=8,
            num_train_epochs=10,                      
            eval_strategy="epoch", 
            save_strategy="epoch",                   
            load_best_model_at_end=True,             
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            report_to="none" 
        )
        
        temp_trainer = WeightedTrainer(
            model=temp_model, args=training_args,
            train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["validation"],
            processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
        )
        
        temp_trainer.train()
        
        val_result = temp_trainer.evaluate()
        val_f1 = val_result['eval_macro_f1']
        
        results_log.append({
            "Rank": r, "LR": lr, 
            "Epoca_Fermata": temp_trainer.state.epoch, 
            "Val_Accuracy": val_result['eval_accuracy'], 
            "Val_Macro_F1": val_f1
        })
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_config = {'r': r, 'lr': lr}

print(f"\n Miglior configurazione: {best_config} (Val Macro F1: {best_f1:.4f})")
display(pd.DataFrame(results_log).sort_values("Val_Macro_F1", ascending=False))


 Avvio Ricerca Iperparametri LoRA sul Validation Set (Con Early Stopping)...

⚙️ Test -> Rank: 4, LR: 0.0001


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.340909,0.272109,0.219591
2,No log,0.446889,0.802721,0.778263
3,No log,0.262285,0.925170,0.920543
4,No log,0.170295,0.959184,0.955748
5,No log,0.136266,0.959184,0.955748
6,0.884554,0.147346,0.959184,0.955748


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.884554,0.170295,6,0.959184,0.955748


...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.181500,0.925170,0.919376
2,No log,0.093466,0.979592,0.978245
3,No log,0.069778,0.965986,0.963892
4,No log,0.078836,0.979592,0.977709


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
No log,0.093466,4,0.979592,0.978245



 Miglior configurazione: {'r': 16, 'lr': 0.0003} (Val Macro F1: 0.9782)


,Rank,LR,Epoca_Fermata,Val_Accuracy,Val_Macro_F1
3,16,0.0003,4.0,0.979592,0.978245
1,4,0.0003,4.0,0.972789,0.971288
2,16,0.0001,8.0,0.972789,0.970845
0,4,0.0001,6.0,0.959184,0.955748


In [59]:
print(f"\n Avvio addestramento finale con Rank {best_config['r']} e LR {best_config['lr']}...")

final_lora_config = LoraConfig(
    r=best_config['r'], 
    lora_alpha=16, 
    lora_dropout=0.1, 
    bias="none", 
    task_type=TaskType.SEQ_CLS,
    target_modules=["query_proj", "key_proj", "value_proj", "output_proj","dense_h_to_4h", "dense_4h_to_h"]
)

final_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
final_model = get_peft_model(final_model, final_lora_config)

final_training_args = TrainingArguments(
    output_dir="./lora_best_model",
    learning_rate=best_config['lr'],
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=15, 
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    weight_decay=0.01,         
    warmup_ratio=0.1,
    report_to=REPORT_TO
)

final_trainer = WeightedTrainer(
    model=final_model, args=final_training_args,
    train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

final_trainer.train()
print(f"Il miglior checkpoint è stato caricato automaticamente in memoria grazie a load_best_model_at_end.")

print("\n Valutazione sul TEST SET con il miglior modello...")
final_test_result = final_trainer.evaluate(eval_dataset=tokenized_ds["test"])

print(f"Risultati sul Test Set -> Accuracy: {final_test_result['eval_accuracy']:.4f} | Macro F1: {final_test_result['eval_macro_f1']:.4f}")


 Avvio addestramento finale con Rank 16 e LR 0.0003...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,4.208703,1.139871,0.367347,0.265881
2,0.524144,0.167166,0.952381,0.949156
3,0.118849,0.078869,0.979592,0.977775
4,0.080175,0.069116,0.986395,0.985156
5,0.037986,0.089527,0.979592,0.977753
6,0.021045,0.080652,0.986395,0.985178
7,0.018078,0.091202,0.972789,0.970312
8,0.006793,0.083654,0.979592,0.977753
9,0.003075,0.080177,0.979592,0.977753


Il miglior checkpoint è stato caricato automaticamente in memoria grazie a load_best_model_at_end.

 Valutazione sul TEST SET con il miglior modello...


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.003075,0.085946,9,0.986395,0.984294


Risultati sul Test Set -> Accuracy: 0.9864 | Macro F1: 0.9843


In [60]:
print("\n Valutazione sul TEST SET...")
final_test_result = final_trainer.evaluate(eval_dataset=tokenized_ds["test"])

print("\n--- CONFRONTO FINALE ---")
result_comparison = {
    "Baseline: Zero-Shot NLI": {"Accuracy": zs_acc, "Macro F1": zs_f1},
    f"LoRA (r={best_config['r']}, lr={best_config['lr']})": {"Accuracy": final_test_result['eval_accuracy'], "Macro F1": final_test_result['eval_macro_f1']}
}
display(pd.DataFrame(result_comparison).T)


print("\n--- ERROR ANALYSIS (Analisi di 3 Fallimenti) ---")


predictions_output = final_trainer.predict(tokenized_ds["test"])
preds = np.argmax(predictions_output.predictions, axis=-1)


refs = np.array(ds["test"]["label"])


errors_idx = np.where(preds != refs)[0]

for i in errors_idx[:3]:
    original = ds["test"][int(i)]
    print("-" * 60)
    print(f"CONTESTO (Premise): {original['premise']}")
    print(f"CLAIM (Hypothesis): {original['hypothesis']}")
    print(f"ETICHETTA VERA: {ID2LABEL[refs[i]]} | PREDETTO DAL MODELLO: {ID2LABEL[preds[i]]}")


if REPORT_TO == "wandb":
    wandb.finish()


 Valutazione sul TEST SET...


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.003075,0.085946,9,0.986395,0.984294



--- CONFRONTO FINALE ---


,Accuracy,Macro F1
Baseline: Zero-Shot NLI,0.897959,0.894452
"LoRA (r=16, lr=0.0003)",0.986395,0.984294



--- ERROR ANALYSIS (Analisi di 3 Fallimenti) ---


------------------------------------------------------------
CONTESTO (Premise): Nel quartiere di Gion a Kyoto, dal 2024 è vietato ai turisti entrare in alcuni vicoli privati; chi viola il divieto rischia una multa di 10000 yen. Le strade pubbliche principali come Hanamikoji restano accessibili.
CLAIM (Hypothesis): Dal 2024 l'intero quartiere di Gion è chiuso ai visitatori.
ETICHETTA VERA: contradiction | PREDETTO DAL MODELLO: entailment
------------------------------------------------------------
CONTESTO (Premise): Le prese giapponesi sono di tipo A, a due lamelle piatte, con corrente a 100 volt: chi arriva dall'Europa deve munirsi di un adattatore.
CLAIM (Hypothesis): Le prese italiane funzionano in Giappone senza adattatore.
ETICHETTA VERA: contradiction | PREDETTO DAL MODELLO: entailment


eval/accuracy,▁██████████
eval/loss,█▂▁▁▁▁▁▁▁▁▁
eval/macro_f1,▁██████████
eval/runtime,▁▅▅█▅▅▅▅▆▆▄
eval/samples_per_second,█▄▄▁▄▄▄▄▃▃▅
eval/steps_per_second,█▄▄▁▄▄▄▄▃▃▅
test/accuracy,▁
test/loss,▁
test/macro_f1,▁
test/runtime,▁
+7,...


In [61]:
import shutil
import os

export_folder = "./fact_checker_lora_model"

print(f"💾 Salvataggio dei pesi LoRA e del Tokenizer in '{export_folder}'...")
final_model.save_pretrained(export_folder)
tokenizer.save_pretrained(export_folder)

zip_filename = "fact_checker_lora"
shutil.make_archive(zip_filename, 'zip', export_folder)
print(f"✅ Archivio creato con successo: {zip_filename}.zip (Dimensione: {os.path.getsize(zip_filename + '.zip') / (1024*1024):.2f} MB)")

💾 Salvataggio dei pesi LoRA e del Tokenizer in './fact_checker_lora_model'...
✅ Archivio creato con successo: fact_checker_lora.zip (Dimensione: 6.66 MB)


In [62]:

print("\n--- SUCCESS ANALYSIS (Esempi classificati correttamente) ---")

logits = predictions_output.predictions
probs = np.exp(logits - logits.max(axis=-1, keepdims=True))
probs = probs / probs.sum(axis=-1, keepdims=True)

correct_idx = np.where(preds == refs)[0]

shown = []
for cls in ID2LABEL:                          
    cand = [i for i in correct_idx if refs[i] == cls]
    if cand:
        best = max(cand, key=lambda i: probs[i][cls])
        shown.append(best)

for i in shown:
    original = ds["test"][int(i)]
    conf = probs[i][preds[i]] * 100
    print("-" * 60)
    print(f"CONTESTO (Premise): {original['premise']}")
    print(f"CLAIM (Hypothesis): {original['hypothesis']}")
    print(f"ETICHETTA VERA: {ID2LABEL[refs[i]]} | PREDETTO: {ID2LABEL[preds[i]]} "
          f"(confidenza: {conf:.1f}%)")

print("-" * 60)
print(f"Totale corretti sul test: {len(correct_idx)}/{len(refs)} "
      f"({len(correct_idx)/len(refs)*100:.1f}%)")


--- SUCCESS ANALYSIS (Esempi classificati correttamente) ---
------------------------------------------------------------
CONTESTO (Premise): In Giappone è obbligatorio togliersi le scarpe prima di entrare nelle case private, nei ryokan e in molti templi; all'ingresso (genkan) si trovano spesso pantofole da indossare all'interno.
CLAIM (Hypothesis): In Giappone si entra tranquillamente in casa con le scarpe.
ETICHETTA VERA: contradiction | PREDETTO: contradiction (confidenza: 100.0%)
------------------------------------------------------------
CONTESTO (Premise): Dall'aeroporto di Narita alla stazione di Tokyo, il Narita Express impiega circa un'ora e costa circa 3070 yen a tratta; la prenotazione del posto è obbligatoria per tutti i passeggeri.
CLAIM (Hypothesis): Un biglietto del Narita Express costa circa 3070 yen.
ETICHETTA VERA: entailment | PREDETTO: entailment (confidenza: 100.0%)
------------------------------------------------------------
CONTESTO (Premise): A Gion, il quarti